In [0]:
### Setup and load the data

In [0]:
from pyspark.sql.functions import *
from pyspark.sql import SparkSession

In [0]:
spark = SparkSession.builder.appName("GlobalSuperStoreAnalytics").getOrCreate()

In [0]:
df = spark.read.option("header", True).option("inferSchema", True).csv("/FileStore/superstore.csv")

In [0]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Market: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Order Priority: string (nullable = true)



In [0]:
### Data cleaning and data conversion

In [0]:
### Converting the data columns into proper format

In [0]:
df = df.withColumn("Sales", col("Sales").cast("double"))

In [0]:
df = df.withColumn("Order Date", to_date("Order Date", "MM/dd/yyyy")) \
       .withColumn("Ship Date", to_date("Ship Date", "MM/dd/yyyy"))

In [0]:
## Creating new columns for analysis

df1 = df.withColumn("Month", date_format("Order Date", "yyyy-MM")) \
    .withColumn("Shipping_Days", datediff("Ship Date","Order Date")) \
    .withColumn("Sales", col("Sales").cast("double")) \
    .withColumn("Profit", col("Profit").cast("double")) \
    .withColumn("Discount", col("Discount").cast("double"))

In [0]:
df1.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Market: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Shipping Cost: double (nullable = true)
 |-- Order Priority: string (nullable = true)
 |-- Month: string (n

In [0]:
## 1 - Identifying monthly sales trend

monthly_sales = df1.groupBy("Month").agg(round(sum("Sales"), 0).alias("Total Sales")).orderBy("Month")
monthly_sales.display()

Month,Total Sales
2012-01,11627.0
2012-02,20635.0
2012-03,13517.0
2012-04,7222.0
2012-05,21909.0
2012-06,23482.0
2012-07,4575.0
2012-08,29971.0
2012-09,31389.0
2012-10,23581.0


In [0]:
## Top 10 cities by Sales

top_cities = df1.groupBy("City").agg(round(sum("Sales"),0).alias("City_Sales")).orderBy(desc("City_Sales")).limit(10)
top_cities.display()

City,City_Sales
Sydney,24379.0
Brisbane,22827.0
Gold Coast,21518.0
Bangkok,21189.0
London,19254.0
Mexico City,18533.0
Perth,16322.0
Delhi,15069.0
Managua,14724.0
Jakarta,14217.0


In [0]:
## Top 10 Most Profitable Product
top_products = df1.groupBy("Product Name").agg(round(sum("Profit"),0).alias("Total_Profit")) \
                 .orderBy(desc("Total_Profit")).limit(10)
top_products.show()


+--------------------+------------+
|        Product Name|Total_Profit|
+--------------------+------------+
|Motorola Smart Ph...|     13088.0|
|   Hoover Stove, Red|     10289.0|
|Cisco Smart Phone...|      9663.0|
|Cisco Smart Phone...|      6607.0|
|Nokia Smart Phone...|      5765.0|
|Harbour Creations...|      5553.0|
|Sauder Classic Bo...|      5285.0|
|Nokia Smart Phone...|      5131.0|
|GBC Ibimaster 500...|      4946.0|
|Safco Classic Boo...|      4905.0|
+--------------------+------------+



In [0]:
## Category wise discount vs profit

category_profit = df1.groupBy("Category").agg(round(avg("Discount"),1).alias("Avg_Discount"),
                                             round(sum("Discount"),0).alias("Total_Profit"))
category_profit.display()

Category,Avg_Discount,Total_Profit
Office Supplies,0.2,42.0
Furniture,0.1,34.0
Technology,0.1,26.0


In [0]:
## Segmentwise Sales

segment_wise_sales = df1.groupBy("Segment").agg(round(sum("Sales"),0).alias("Segment_Sales"))
segment_wise_sales.show()

+-----------+-------------+
|    Segment|Segment_Sales|
+-----------+-------------+
|   Consumer|     873394.0|
|Home Office|     313143.0|
|  Corporate|     524271.0|
+-----------+-------------+



In [0]:
#High discount and  NegativeProfits
discount_loss = df1.filter((col("Discount") > 0.4) & (col("Profit") < 0)) \
                  .select("Product Name", "Discount", "Profit").distinct()
discount_loss.show(10)

+--------------------+--------+--------+
|        Product Name|Discount|  Profit|
+--------------------+--------+--------+
|StarTech Printer,...|     0.5| -286.92|
|Cisco CP-7937G Un...|     0.5|  -27.83|
|Avery Metallic Po...|     0.7|    -6.3|
|Lesro Wood Table,...|    0.47| -140.72|
|"Avery Trapezoid ...|     0.7|  -58.72|
|Office Star Execu...|     0.5|-2211.17|
|Avery Recycled Fl...|     0.7|  -21.16|
| Hoover Stove, White|     0.5| -1784.9|
|Bevis Conference ...|    0.47| -452.81|
|Apple Smart Phone...|     0.5|-1783.08|
+--------------------+--------+--------+
only showing top 10 rows



In [0]:
## Repeat customers more than 3 orders
repeat_customers = df1.groupBy("Customer Name").agg(countDistinct("Order ID").alias("Order_Count")) \
    .filter("Order_Count > 3").orderBy(desc("Order_Count"))
repeat_customers.display()

Customer Name,Order_Count
Adrian Barton,13
Adam Hart,13
Alan Hwang,11
Aaron Hawkins,10
Adam Bellavance,10
Adam Shillingsburg,10
Alan Barnes,9
Alan Dominguez,9
Aaron Smayling,7
Adrian Hane,7


In [0]:
##Regionwise Profitability
region_profit = df.groupBy("Region").agg(round(sum("Profit"),0).alias("Region_profit"))
region_profit.display()

Region,Region_profit
Western US,2450.0
Eastern Europe,12228.0
Central US,5674.0
Southern Europe,20558.0
Eastern US,2537.0
Central America,21742.0
North Africa,7416.0
Northern Europe,23553.0
Caribbean,5464.0
Western Africa,1278.0
